# 6.1 Episodic Generalization Optimization - EGO

## Introduction

Human cognition is unique in its ability to perform a wide range of tasks and to learn new tasks quickly. Both abilities have long been associated with the acquisition of knowledge that can generalize across tasks and the flexible use of that knowledge to execute goal-directed behavior. In this tutorial, we introduce how this can emerge in a neural network by implementing the Episodic Generalization and Optimization (EGO) framework. The framework consists of an episodic memory module, which rapidly learns relationships between stimuli; a semantic pathway, which more slowly learns how stimuli map to responses; and a recurrent context module, which maintains a representation of task-relevant context information, integrates this over time, and uses it to recall context-relevant memories.

![EGO](https://princetonuniversity.github.io/NEU-PSY-502/_static/images/502B/computation/em/ego_model.png)

The EGO framework consists of a control mechanism (context module; upper middle) and an episodic memory mechanism (bottom left). Episodic memory records conjunctions of stimuli (blue boxes), contexts (pink boxes), and observed responses (green boxes) at each time point (rows). Bidirectional arrows connect episodic memory to the stimulus, context, and output, indicating that these values can be stored in or used to query episodic memory, or retrieved from it when another field is queried. You can think of this as a more flexible dictionary that stores triplets instead of distinct key-value pairs, and allows any field (or any combinations of fields) to act as a key. The context module integrates previous context (recurrent connection) along with information about the stimulus and the context retrieved from memory.


Here we show that the EGO framework can emulate human behavior in a specific learning environment where participants are trained on two sets of sequences involving identical states presented in different orders for different contexts. Empirical findings show that participants perform better when trained in blocks of each context than when trained interleaved:

### Task: Coffe Shop World (CSW)

<p align="center">
    <img src="https://princetonuniversity.github.io/NEU-PSY-502/_static/images/502B/computation/em/suspicious.png" alt="suspicious barrista" style="width:45%; margin-right:10px;">
    <img src="https://princetonuniversity.github.io/NEU-PSY-502/_static/images/502B/computation/em/gratitude.png" alt="caffe graditude" style="width:45%;">
</p>


Imagine, you are in a city with two coffee shops, each with a different layout and different ways of ordering. In one coffee shop—called *The Suspicious Barista*—you order first, pay for the coffee, and then sit down to wait until the waiter brings your order. In the other coffee shop—called *Café Gratitude*—you sit down first, wait until the waiter comes and takes your order. You pay after finishing the coffee.

This example demonstrates that many situations share similar stimuli but have different transition structures. Simple integration will help the system learn the transition structure, but it will only provide a weak cue about the difference between them due to the similarity between the situations. In other words the states --ordering, paying, and sitting down-- are very similar between the two situations and are therefore hard to distinguish. This can be overcome by differentiating the context representations associated with each setting (e.g., learning different context representations for coffee shops with paranoid vs. gullible baristas). Recent empirical work suggests that people can learn how to do this very effectively, but that this depends on the temporal structure of the environment: people do better when trained in blocks of each situation than when trained interleaved ([Beukers et al., 2023](https://www.nature.com/articles/s44271-024-00079-4)).

We start with creating a dataset for the CSW task.

**Installation and Setup**


## Generating data for the CSW task

We start by generating a dataset for the CSW task. The dataset consists of sequences of states. The task is to predict the next state given the current state and the context. The transition between states is determined by the context which in turn is determined by the "first" state in the sequence. The following figure illustrates the task structure:

![EGO](https://princetonuniversity.github.io/NEU-PSY-502/_static/images/502B/computation/em/csw.png)



In [4]:
import random

import numpy as np
from torch.utils.data import dataset
import torch
from random import randint

def one_hot_encode(labels, num_classes):
    '''
    One hot encode labels and convert to tensor.
    '''
    return torch.tensor((np.arange(num_classes) == labels[..., None]).astype(float),dtype=torch.float32)

class CSWDataset(dataset.Dataset):
    """
    A custom dataset class for generating samples based on different contexts.

    Args:
        n_samples_per_context (list): A list of integers representing the number of samples to generate for each context.
        contexts_to_load (list): A list of integers representing the contexts to load.
        probs (list, optional): A list of probabilities for generating the samples. Defaults to [1, 1, 1].
    """

    def __init__(self, n_samples_per_context, contexts_to_load, probs=[1, 1, 1]) -> None:
        super().__init__()

        self.n_samples_per_context = n_samples_per_context
        self.all_trials = []

        for i, context in enumerate(contexts_to_load):
            for _ in range(n_samples_per_context[i]):
                if context == 0:
                    self.all_trials.extend(self.gen_context1(probs))
                else:
                    self.all_trials.extend(self.gen_context2(probs))

        self.xs = one_hot_encode(np.array(self.all_trials), 11)
        self.xs = self.xs.reshape((-1, 11))
        self.ys = torch.cat([self.xs[1:], one_hot_encode(np.array([0]), 11)], dim=0)

        # Remove the last transition since there's no next state available
        self.xs = self.xs[:-1]
        self.ys = self.ys[:-1]
        self.contexts = self.xs

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, idx):
        return self.xs[idx], self.contexts[idx], self.ys[idx]

    def gen_context1(self, probs):
        """
        Generate samples for context 1 based on the given probabilities.

        Args:
            probs (list): A list of probabilities for generating the samples.

        Returns:
            list: A list of states representing the generated samples.
        """
        states = [9, random.choice([1, 2])]
        for p in probs:
            if random.random() <= p:
                states.append(states[-1] + 2)
            else:
                if states[-1] % 2 == 0:
                    states.append(states[-1] + 1)
                else:
                    states.append(states[-1] + 3)
        return states

    def gen_context2(self, probs):
        """
        Generate samples for context 2 based on the given probabilities.

        Args:
            probs (list): A list of probabilities for generating the samples.

        Returns:
            list: A list of states representing the generated samples.
        """
        states = [10, random.choice([1, 2])]
        for p in probs:
            if random.random() <= p:
                if states[-1] % 2 == 0:
                    states.append(states[-1] + 1)
                else:
                    states.append(states[-1] + 3)
            else:
                states.append(states[-1] + 2)
        return states

def gen_data_loader(paradigm, probs=[1., 1., 1.], n=1):
    if paradigm == 'tst':
        contexts_to_load =[0]
        n_samples_per_context = [n]
        ds = CSWDataset(n_samples_per_context, contexts_to_load, probs=[1.])

    if paradigm == 'blocked':
        contexts_to_load = [0, 1, 0, 1] + [randint(0, 2) for _ in range(n)]
        n_samples_per_context = [n, n, n, n] + [1] * n
        ds = CSWDataset(n_samples_per_context, contexts_to_load, probs=probs)
    elif paradigm == 'interleaved':
        contexts_to_load = [0, 1] * (2 * n) + [randint(0, 2) for _ in range(n)]
        n_samples_per_context = [1] * (4 * n) + [1] * n
        ds = CSWDataset(n_samples_per_context, contexts_to_load, probs=probs)
    return torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)

print('Generating data...')
data_loader = gen_data_loader('tst', [1., 1., 1.], 2)

ego_inputs = data_loader.dataset.xs.numpy()
ego_targets = data_loader.dataset.ys.numpy()

print(ego_inputs)
print(ego_targets)

print(type(ego_inputs))
print(type(ego_targets))

print(ego_inputs.shape)
print(ego_targets.shape)

Generating data...
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]]
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(5, 11)
(5, 11)


In [3]:
params_ego = dict(

    # Names:
    name="EGO Model CSW",
    em_name="EM",
    state_input_layer_name="STATE",
    previous_state_layer_name="PREVIOUS STATE",
    context_layer_name='CONTEXT',
    prediction_layer_name="PREDICTION",

    # Structural
    state_d=11,  # length of state vector
    previous_state_d=11,  # length of state vector
    context_d=11,  # length of context vector
    memory_capacity=ALL,  # number of entries in EM memory; ALL=> match to number of stims
    memory_init=.001,  # Initialize memory with random values in interval
    concatenate_queries=False,  # whether to concatenate queries before retrieval

    # environment
    curriculum_type='Blocked',
    num_stims=ALL,  # Integer or ALL

    # Processing
    integration_rate=.5,  # don't change this
    previous_state_weight=1.,  # weight of the state used during memory retrieval
    context_weight=1.,  # weight of the context used during memory retrieval
    state_weight=None,  # weight of the state used during memory retrieval
    normalize_field_weights=False,  # whether to normalize the field weights during memory retrieval
    normalize_memories=False,  # whether to normalize the memory during memory retrieval
    softmax_temperature=1.,  # temperature of the softmax used during memory retrieval (smaller means more argmax-like
    softmax_threshold=.01,  # threshold used to mask out small values in softmax
    enable_learning=True,
    loss_spec=Loss.BINARY_CROSS_ENTROPY,
    learning_rate=0.,
    num_optimization_steps=1,
    synch_weights=RUN,
    synch_values=RUN,
    synch_results=RUN,
    execution_mode=ExecutionMode.PyTorch,
    device=CPU,
)

def construct_model(
        memory_capacity,
        model_name: str = params_ego['name'],

        # Input layer:
        state_input_name: str = params_ego['state_input_layer_name'],
        state_size: int = params_ego['state_d'],

        # Previous state
        previous_state_name: str = params_ego['previous_state_layer_name'],

        # Context representation (learned):
        context_name: str = params_ego['context_layer_name'],
        context_size: Union[float, int] = params_ego['context_d'],
        integration_rate: float = params_ego['integration_rate'],

        # EM:
        em_name: str = params_ego['em_name'],
        retrieval_softmax_gain=retrieval_softmax_gain,
        retrieval_softmax_threshold=params_ego['softmax_threshold'],
        state_retrieval_weight: Union[float, int] = params_ego['state_weight'],
        previous_state_retrieval_weight: Union[float, int] = params_ego['previous_state_weight'],
        context_retrieval_weight: Union[float, int] = params_ego['context_weight'],
        normalize_field_weights=params_ego['normalize_field_weights'],
        concatenate_queries=params_ego['concatenate_queries'],
        enable_learning=params_ego['enable_learning'],

        memory_init=params_ego['memory_init'],

        # Output:
        prediction_layer_name: str = params_ego['prediction_layer_name'],

        # Learning
        loss_spec=params_ego['loss_spec'],
        # target_fields=params_ego['target_fields'],
        learning_rate=params_ego['learning_rate'],
        device=params_ego['device']

) -> Composition:
    assert 0 <= integration_rate <= 1, \
        f"integrator_retrieval_weight must be a number from 0 to 1"

    # ----------------------------------------------------------------------------------------------------------------
    # -------------------------------------------------  Nodes  ------------------------------------------------------
    # ----------------------------------------------------------------------------------------------------------------

    state_input_layer = ProcessingMechanism(name=state_input_name, input_shapes=state_size)
    previous_state_layer = ProcessingMechanism(name=previous_state_name, input_shapes=state_size)
    context_layer = TransferMechanism(name=context_name,
                                      input_shapes=context_size,
                                      function=Tanh,
                                      integrator_mode=True,
                                      integration_rate=integration_rate)


    em = EMComposition(name=em_name,
                       memory_template=(3, state_size),  # context
                       memory_fill=memory_init,
                       memory_capacity=memory_capacity,
                       normalize_memories=False,
                       normalize_field_weights=normalize_field_weights,
                       memory_decay_rate=0,
                       softmax_gain=retrieval_softmax_gain,
                       softmax_threshold=retrieval_softmax_threshold,
                       fields={state_input_name: {FIELD_WEIGHT: state_retrieval_weight,
                                                  LEARN_FIELD_WEIGHT: False,
                                                  TARGET_FIELD: True},
                               previous_state_name: {FIELD_WEIGHT: previous_state_retrieval_weight,
                                                     LEARN_FIELD_WEIGHT: False,
                                                     TARGET_FIELD: False},
                               context_name: {FIELD_WEIGHT: context_retrieval_weight,
                                              LEARN_FIELD_WEIGHT: False,
                                              TARGET_FIELD: False}},

                       enable_learning=enable_learning,
                       learning_rate=learning_rate,
                       # device=device,
                       )

    prediction_layer = ProcessingMechanism(name=prediction_layer_name, input_shapes=state_size)

    # ----------------------------------------------------------------------------------------------------------------
    # -------------------------------------------------  EGO Composition  --------------------------------------------
    # ----------------------------------------------------------------------------------------------------------------

    QUERY = ' [QUERY]'
    VALUE = ' [VALUE]'
    RETRIEVED = ' [RETRIEVED]'

    # Pathways
    state_to_previous_state_pathway = [state_input_layer,
                                       MappingProjection(matrix=IDENTITY_MATRIX,
                                                         learnable=False),
                                       previous_state_layer]
    state_to_context_pathway = [state_input_layer,
                                MappingProjection(matrix=IDENTITY_MATRIX,
                                                  learnable=False),
                                context_layer]
    state_to_em_pathway = [state_input_layer,
                           MappingProjection(sender=state_input_layer,
                                             receiver=em.nodes[state_input_name + VALUE],
                                             matrix=IDENTITY_MATRIX,
                                             learnable=False),
                           em]
    previous_state_to_em_pathway = [previous_state_layer,
                                    MappingProjection(sender=previous_state_layer,
                                                      receiver=em.nodes[previous_state_name + QUERY],
                                                      matrix=IDENTITY_MATRIX,
                                                      learnable=False),
                                    em]
    context_learning_pathway = [context_layer,
                                MappingProjection(sender=context_layer,
                                                  matrix=IDENTITY_MATRIX,
                                                  receiver=em.nodes[context_name + QUERY],
                                                  learnable=True),
                                em,
                                MappingProjection(sender=em.nodes[state_input_name + RETRIEVED],
                                                  receiver=prediction_layer,
                                                  matrix=IDENTITY_MATRIX,
                                                  learnable=False),
                                prediction_layer]

    # Composition
    EGO_comp = AutodiffComposition([state_to_previous_state_pathway,
                                    state_to_context_pathway,
                                    state_to_em_pathway,
                                    previous_state_to_em_pathway,
                                    context_learning_pathway],
                                   learning_rate=learning_rate,
                                   loss_spec=loss_spec,
                                   name=model_name,
                                   device=device)

    learning_components = EGO_comp.infer_backpropagation_learning_pathways(ExecutionMode.PyTorch)
    EGO_comp.add_projection(MappingProjection(sender=state_input_layer,
                                              receiver=learning_components[0],
                                              learnable=False))

    EGO_comp.scheduler.add_condition(em, BeforeNodes(previous_state_layer, context_layer))

    return EGO_comp, context_layer, state_input_layer, em


def run_model(model,
              context_layer,
              state_input_layer,
              em,
              trials,
              learning=True,
              ):
    if learning:
        for t in trials:
            model.learn(inputs={params_ego['state_input_layer_name']: [t]},
                        learning_rate=params_ego['learning_rate'],
                        execution_mode=params_ego['execution_mode'],
                        minibatch_size=1,
                        call_before_minibatch=hi(model, context_layer, state_input_layer, em)
                        )
        # model.learn(inputs={params_ego['state_input_layer_name']: trials},
        #             # learning_rate=params_ego['learning_rate'],
        #             execution_mode=params_ego['execution_mode'],
        #             synch_projection_matrices_with_torch=params_ego['synch_weights'],
        #             synch_node_values_with_torch=params_ego['synch_values'],
        #             synch_results_with_torch=params_ego['synch_results'],
        #             minibatch_size=1,
        #             )
        memory = em.memory
        print(memory)
    return model.results[:, 2]

data_loader = gen_data_loader(TRAINING_PARADIGM, PROBS, 2)
fig, axes = plt.subplots(2, 1, figsize=(5, 12))

ego_inputs = data_loader.dataset.xs.numpy()
ego_targets = data_loader.dataset.ys.numpy()

model, context, state, em = construct_model(memory_capacity=len(ego_inputs))

plot_results(ego_results, ego_targets, axes[0], 0)

NameError: name 'ALL' is not defined